# Hierarchical Cell Type Classification Training

Two training modes:
- **Mode A**: Precomputed weighted-sum embeddings + SimpleNN classifier
- **Mode B**: End-to-end attention pooling + WideNN classifier

Both use MarginalizationLoss with blood cell hierarchy (CL:0000988).

**Runs on cluster** (needs SOMA database + ESM-2 embeddings).

## 1. Setup & Data Loading

In [ ]:
import sys
import logging
import pickle
import time
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from torch.nn.utils import clip_grad_norm_
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
import matplotlib.pyplot as plt

# Find project root by searching upward for pyproject.toml
_cwd = Path(".").resolve()
PROJECT_ROOT = _cwd
for _parent in [_cwd] + list(_cwd.parents):
    if (_parent / "pyproject.toml").exists():
        PROJECT_ROOT = _parent
        break
sys.path.insert(0, str(PROJECT_ROOT))
DATA_DIR = PROJECT_ROOT / "data"
print(f"Project root: {PROJECT_ROOT}")

from scipher.hierarchy import (
    load_prebuilt_hierarchy, MarginalizationLoss, SimpleNN, WideNN,
)
from scipher.embedders.weighted_sum import WeightedSumEmbedder
from scipher.embedders.attention import AttentionPooling, CellDataset

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger(__name__)

# --- Config ---
DATE = "2026-01-29"
ROOT_CL_ID = "CL:0000988"  # blood cells
SOMA_URI = "/scratch/sigbio_project_root/sigbio_project25/jingqiao/mccell-single/soma_db_homo_sapiens"
N_CELLS = 200_000
MIN_CELL_COUNT = 50
BATCH_SIZE = 256
LR = 1e-4
LEAF_WEIGHT = 7.0
GRAD_CLIP = 1.0
EPOCHS_A = 15
EPOCHS_B = 10
SEED = 42
NUM_WORKERS = 8

run_timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
num_gpus = torch.cuda.device_count()
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPUs: {num_gpus}x {torch.cuda.get_device_name(0)}")
    print(f"Effective batch size: {BATCH_SIZE * num_gpus} (BATCH_SIZE={BATCH_SIZE} x {num_gpus} GPUs)")

In [ ]:
(
    mapping_dict, leaf_values, internal_values,
    marginalization_df, parent_child_df, exclusion_df,
) = load_prebuilt_hierarchy(DATE, ROOT_CL_ID)

all_cell_values = list(leaf_values) + list(internal_values)
print(f"Leaf types: {len(leaf_values)}, Internal: {len(internal_values)}, Total: {len(all_cell_values)}")

In [ ]:
EMB_PATH = DATA_DIR / "embeddings" / "gene_to_embedding.pkl"
with open(EMB_PATH, "rb") as f:
    gene_to_embedding = pickle.load(f)

embed_dim = next(iter(gene_to_embedding.values())).shape[0]
print(f"Gene embeddings: {len(gene_to_embedding):,} genes x {embed_dim}-dim")

In [ ]:
BIOMART_FILE = DATA_DIR / "raw" / "mart_export.txt"
df_biomart = pd.read_csv(BIOMART_FILE)
df_pc = df_biomart[df_biomart["Gene type"] == "protein_coding"].dropna(
    subset=["Gene stable ID", "Gene name"]
)
ensembl_to_symbol = dict(zip(df_pc["Gene stable ID"], df_pc["Gene name"]))
gene_list = df_pc["Gene stable ID"].unique().tolist()
print(f"Ensembl->symbol mappings: {len(ensembl_to_symbol):,}")

In [ ]:
import tiledbsoma as soma

experiment = soma.open(SOMA_URI, mode="r")

# Step 1: Read obs metadata
obs_df = (
    experiment.obs.read(
        value_filter='assay == "10x 3\' v3" and is_primary_data == True',
        column_names=["soma_joinid", "cell_type_ontology_term_id", "cell_type"],
    )
    .concat()
    .to_pandas()
)
print(f"Total 10x v3 primary cells: {len(obs_df):,}")

# Filter to hierarchy cell types
obs_df = obs_df[obs_df["cell_type_ontology_term_id"].isin(all_cell_values)].copy()
print(f"Blood cells in hierarchy: {len(obs_df):,}")

# Drop rare cell types
type_counts = obs_df["cell_type_ontology_term_id"].value_counts()
keep_types = type_counts[type_counts >= MIN_CELL_COUNT].index
obs_df = obs_df[obs_df["cell_type_ontology_term_id"].isin(keep_types)].copy()
print(
    f"After dropping < {MIN_CELL_COUNT} cells: {len(obs_df):,} cells, "
    f"{obs_df['cell_type_ontology_term_id'].nunique()} types"
)

# Subsample
if len(obs_df) > N_CELLS:
    obs_df = obs_df.sample(n=N_CELLS, random_state=SEED)
    print(f"Subsampled to {len(obs_df):,} cells")

# Step 2: Load expression for selected cells
var_value_filter = f"feature_id in {gene_list}"
joinids = obs_df["soma_joinid"].tolist()

with experiment.axis_query(
    measurement_name="RNA",
    obs_query=soma.AxisQuery(coords=(joinids,)),
    var_query=soma.AxisQuery(value_filter=var_value_filter),
) as query:
    adata = query.to_anndata(
        X_name="raw",
        column_names={"obs": ["cell_type_ontology_term_id", "cell_type"]},
    )

print(f"\nLoaded AnnData: {adata.shape[0]:,} cells x {adata.shape[1]:,} genes")
print(f"Cell types: {adata.obs['cell_type_ontology_term_id'].nunique()}")
print(f"\nDistribution (top 15):")
print(adata.obs["cell_type"].value_counts().head(15).to_string())

In [ ]:
# Remap Ensembl IDs to gene symbols
original_var_names = adata.var_names.tolist()
new_var_names = [ensembl_to_symbol.get(eid, eid) for eid in original_var_names]

# Deduplicate: keep first occurrence
seen = set()
dup_mask = []
for name in new_var_names:
    if name in seen:
        dup_mask.append(False)
    else:
        seen.add(name)
        dup_mask.append(True)

n_dups = sum(1 for x in dup_mask if not x)
if n_dups > 0:
    adata = adata[:, dup_mask].copy()
    new_var_names = [n for n, keep in zip(new_var_names, dup_mask) if keep]

adata.var_names = new_var_names
adata.var_names_make_unique()

n_mapped = sum(1 for name in adata.var_names if name in gene_to_embedding)
print(f"Genes with embeddings: {n_mapped:,}/{adata.n_vars:,} ({100*n_mapped/adata.n_vars:.1f}%)")
print(f"Duplicates removed: {n_dups:,}")

In [ ]:
# Integer labels via mapping_dict
labels = np.array([mapping_dict[ct] for ct in adata.obs["cell_type_ontology_term_id"]])

# Reverse mappings for evaluation
idx_to_cl = {v: k for k, v in mapping_dict.items()}
cl_to_name = (
    adata.obs.drop_duplicates("cell_type_ontology_term_id")
    .set_index("cell_type_ontology_term_id")["cell_type"]
    .to_dict()
)
leaf_idx_set = {mapping_dict[cl] for cl in leaf_values}

# Stratified train/val split
train_idx, val_idx = train_test_split(
    np.arange(len(labels)),
    test_size=0.2,
    random_state=SEED,
    stratify=adata.obs["cell_type_ontology_term_id"],
)

train_adata = adata[train_idx].copy()
val_adata = adata[val_idx].copy()
train_labels = labels[train_idx]
val_labels = labels[val_idx]

n_train_leaf = sum(1 for l in train_labels if l in leaf_idx_set)
n_val_leaf = sum(1 for l in val_labels if l in leaf_idx_set)
print(f"Train: {len(train_labels):,} cells ({n_train_leaf:,} leaf-labeled)")
print(f"Val:   {len(val_labels):,} cells ({n_val_leaf:,} leaf-labeled)")

## 2. Mode A -- Precomputed Weighted-Sum + Classifier

In [ ]:
embedder = WeightedSumEmbedder(gene_to_embedding)
train_embeddings = embedder.embed(train_adata)
val_embeddings = embedder.embed(val_adata)
print(f"Train embeddings: {train_embeddings.shape}")
print(f"Val embeddings:   {val_embeddings.shape}")

In [ ]:
train_ds_a = TensorDataset(
    torch.tensor(train_embeddings, dtype=torch.float32),
    torch.tensor(train_labels, dtype=torch.long),
)
val_ds_a = TensorDataset(
    torch.tensor(val_embeddings, dtype=torch.float32),
    torch.tensor(val_labels, dtype=torch.long),
)
train_loader_a = DataLoader(
    train_ds_a, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True,
)
val_loader_a = DataLoader(
    val_ds_a, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
)
print(f"Train batches: {len(train_loader_a)}, Val batches: {len(val_loader_a)}")

In [ ]:
model_a = SimpleNN(input_dim=embed_dim, output_dim=len(leaf_values)).to(device)
if num_gpus > 1:
    model_a = nn.DataParallel(model_a)
    print(f"Using DataParallel across {num_gpus} GPUs")

loss_fn_a = MarginalizationLoss(
    marginalization_df, parent_child_df, exclusion_df,
    leaf_values, internal_values, mapping_dict,
    leaf_weight=LEAF_WEIGHT, device=device,
)
optimizer_a = optim.Adam(model_a.parameters(), lr=LR)

n_params = sum(p.numel() for p in model_a.parameters() if p.requires_grad)
print(f"SimpleNN: {n_params:,} trainable parameters")
print(f"Input: {embed_dim} -> Output: {len(leaf_values)} leaf classes")

In [ ]:
loss_history_a = {"total": [], "leaf": [], "parent": []}
epoch_times_a = []

print(f"Training Mode A: {EPOCHS_A} epochs, {len(train_loader_a)} batches/epoch")
print("-" * 70)

for epoch in range(EPOCHS_A):
    model_a.train()
    epoch_start = time.time()
    epoch_losses = []

    for i, (X_batch, y_batch) in enumerate(train_loader_a):
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer_a.zero_grad()
        logits = model_a(X_batch)
        total_loss, loss_leafs, loss_parents = loss_fn_a(logits, y_batch)
        total_loss.backward()
        clip_grad_norm_(model_a.parameters(), max_norm=GRAD_CLIP)
        optimizer_a.step()

        loss_history_a["total"].append(total_loss.item())
        loss_history_a["leaf"].append(loss_leafs.item())
        loss_history_a["parent"].append(loss_parents.item())
        epoch_losses.append(total_loss.item())

        if (i + 1) % 200 == 0:
            avg_recent = np.mean(epoch_losses[-200:])
            print(
                f"  [Epoch {epoch+1}/{EPOCHS_A}, Batch {i+1}/{len(train_loader_a)}] "
                f"Loss: {total_loss.item():.4f} (avg: {avg_recent:.4f})"
            )

    epoch_time = time.time() - epoch_start
    epoch_times_a.append(epoch_time)
    avg_loss = np.mean(epoch_losses)
    print(f"Epoch {epoch+1}/{EPOCHS_A} -- avg loss: {avg_loss:.4f}, time: {epoch_time:.1f}s")

In [ ]:
model_a.eval()
all_preds_a, all_labels_a = [], []

with torch.no_grad():
    for X_batch, y_batch in val_loader_a:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        logits = model_a(X_batch)
        preds = torch.argmax(logits, dim=1)

        # Keep only leaf-labeled samples for accuracy
        is_leaf = torch.tensor(
            [y.item() in leaf_idx_set for y in y_batch], device=device
        )
        if is_leaf.sum() > 0:
            all_preds_a.extend(preds[is_leaf].cpu().numpy())
            all_labels_a.extend(y_batch[is_leaf].cpu().numpy())

all_preds_a = np.array(all_preds_a)
all_labels_a = np.array(all_labels_a)

acc_a = (all_preds_a == all_labels_a).mean()
macro_f1_a = f1_score(all_labels_a, all_preds_a, average="macro")
weighted_f1_a = f1_score(all_labels_a, all_preds_a, average="weighted")

print(f"=== Mode A Validation (leaf-labeled cells) ===")
print(f"  Leaf accuracy: {acc_a:.4f}")
print(f"  Macro F1:      {macro_f1_a:.4f}")
print(f"  Weighted F1:   {weighted_f1_a:.4f}")
print(f"  N samples:     {len(all_labels_a):,}")

# Per-class accuracy
rows = []
for idx in sorted(set(all_labels_a)):
    cl_id = idx_to_cl[idx]
    name = cl_to_name.get(cl_id, cl_id)
    mask = all_labels_a == idx
    n = mask.sum()
    acc = (all_preds_a[mask] == all_labels_a[mask]).mean()
    rows.append({"cell_type": name, "cl_id": cl_id, "n": n, "accuracy": acc})

df_eval_a = pd.DataFrame(rows).sort_values("accuracy")
print(f"\nPer-class accuracy:")
print(df_eval_a.to_string(index=False, float_format="{:.3f}".format))

In [ ]:
checkpoint_dir = DATA_DIR / "checkpoints"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

checkpoint_path_a = checkpoint_dir / f"mode_a_{run_timestamp}.pt"
state_dict_a = model_a.module.state_dict() if isinstance(model_a, nn.DataParallel) else model_a.state_dict()
checkpoint_a = {
    "epoch": EPOCHS_A,
    "model_state_dict": state_dict_a,
    "optimizer_state_dict": optimizer_a.state_dict(),
    "loss_history": loss_history_a,
    "epoch_times": epoch_times_a,
    "input_dim": embed_dim,
    "output_dim": len(leaf_values),
    "date_preprocessed": DATE,
    "root_cl_id": ROOT_CL_ID,
    "model_class": "SimpleNN",
    "leaf_accuracy": acc_a,
    "macro_f1": macro_f1_a,
    "num_gpus": num_gpus,
}
torch.save(checkpoint_a, checkpoint_path_a)
print(f"Saved Mode A checkpoint: {checkpoint_path_a}")

## 3. Mode B -- End-to-End Attention Pooling + Classifier

In [ ]:
class ScipherModel(nn.Module):
    def __init__(self, embed_dim, num_leaves, hidden_dim=256):
        super().__init__()
        self.embedder = AttentionPooling(embed_dim=embed_dim, hidden_dim=hidden_dim)
        self.classifier = WideNN(input_dim=embed_dim, output_dim=num_leaves)

    def forward(self, gene_embeddings, expression, mask):
        cell_embedding, attn_weights = self.embedder(gene_embeddings, expression, mask)
        logits = self.classifier(cell_embedding)
        return logits, cell_embedding, attn_weights

In [ ]:
train_dataset_b = CellDataset([train_adata], train_labels, gene_to_embedding)
val_dataset_b = CellDataset([val_adata], val_labels, gene_to_embedding)

train_loader_b = DataLoader(
    train_dataset_b, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True,
)
val_loader_b = DataLoader(
    val_dataset_b, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
)
print(f"Train batches: {len(train_loader_b)}, Val batches: {len(val_loader_b)}")

In [ ]:
model_b = ScipherModel(
    embed_dim=embed_dim, num_leaves=len(leaf_values), hidden_dim=256,
).to(device)
if num_gpus > 1:
    model_b = nn.DataParallel(model_b)
    print(f"Using DataParallel across {num_gpus} GPUs")

loss_fn_b = MarginalizationLoss(
    marginalization_df, parent_child_df, exclusion_df,
    leaf_values, internal_values, mapping_dict,
    leaf_weight=LEAF_WEIGHT, device=device,
)
optimizer_b = optim.Adam(model_b.parameters(), lr=LR)

n_params = sum(p.numel() for p in model_b.parameters() if p.requires_grad)
print(f"ScipherModel: {n_params:,} trainable parameters")
attn_params = sum(p.numel() for n, p in model_b.named_parameters() if "embedder" in n)
cls_params = sum(p.numel() for n, p in model_b.named_parameters() if "classifier" in n)
print(f"  AttentionPooling: {attn_params:,}")
print(f"  WideNN classifier: {cls_params:,}")

In [ ]:
# Load gene embeddings once (shared across all batches)
gene_embs = train_dataset_b.get_gene_embeddings(device)

loss_history_b = {"total": [], "leaf": [], "parent": []}
epoch_times_b = []

print(f"Training Mode B: {EPOCHS_B} epochs, {len(train_loader_b)} batches/epoch")
print("-" * 70)

for epoch in range(EPOCHS_B):
    model_b.train()
    epoch_start = time.time()
    epoch_losses = []

    for i, batch in enumerate(train_loader_b):
        expression = batch["expression"].to(device)
        mask = batch["mask"].to(device)
        y_batch = batch["label"].to(device)

        # Expand gene embeddings to batch size (zero-copy view)
        batch_size = expression.size(0)
        embeddings = gene_embs.unsqueeze(0).expand(batch_size, -1, -1)

        optimizer_b.zero_grad()
        logits, _, _ = model_b(embeddings, expression, mask)
        total_loss, loss_leafs, loss_parents = loss_fn_b(logits, y_batch)
        total_loss.backward()
        clip_grad_norm_(model_b.parameters(), max_norm=GRAD_CLIP)
        optimizer_b.step()

        loss_history_b["total"].append(total_loss.item())
        loss_history_b["leaf"].append(loss_leafs.item())
        loss_history_b["parent"].append(loss_parents.item())
        epoch_losses.append(total_loss.item())

        if (i + 1) % 200 == 0:
            avg_recent = np.mean(epoch_losses[-200:])
            print(
                f"  [Epoch {epoch+1}/{EPOCHS_B}, Batch {i+1}/{len(train_loader_b)}] "
                f"Loss: {total_loss.item():.4f} (avg: {avg_recent:.4f})"
            )

    epoch_time = time.time() - epoch_start
    epoch_times_b.append(epoch_time)
    avg_loss = np.mean(epoch_losses)
    print(f"Epoch {epoch+1}/{EPOCHS_B} -- avg loss: {avg_loss:.4f}, time: {epoch_time:.1f}s")

In [ ]:
# Validation
gene_embs_val = val_dataset_b.get_gene_embeddings(device)
model_b.eval()
all_preds_b, all_labels_b = [], []
val_cell_embeddings = []

with torch.no_grad():
    for batch in val_loader_b:
        expression = batch["expression"].to(device)
        mask = batch["mask"].to(device)
        y_batch = batch["label"].to(device)

        batch_size = expression.size(0)
        embeddings = gene_embs_val.unsqueeze(0).expand(batch_size, -1, -1)

        logits, cell_emb, _ = model_b(embeddings, expression, mask)
        preds = torch.argmax(logits, dim=1)

        is_leaf = torch.tensor(
            [y.item() in leaf_idx_set for y in y_batch], device=device
        )
        if is_leaf.sum() > 0:
            all_preds_b.extend(preds[is_leaf].cpu().numpy())
            all_labels_b.extend(y_batch[is_leaf].cpu().numpy())

        val_cell_embeddings.append(cell_emb.cpu().numpy())

all_preds_b = np.array(all_preds_b)
all_labels_b = np.array(all_labels_b)
val_embeddings_b = np.concatenate(val_cell_embeddings, axis=0)

acc_b = (all_preds_b == all_labels_b).mean()
macro_f1_b = f1_score(all_labels_b, all_preds_b, average="macro")
weighted_f1_b = f1_score(all_labels_b, all_preds_b, average="weighted")

print(f"=== Mode B Validation (leaf-labeled cells) ===")
print(f"  Leaf accuracy: {acc_b:.4f}")
print(f"  Macro F1:      {macro_f1_b:.4f}")
print(f"  Weighted F1:   {weighted_f1_b:.4f}")
print(f"  N samples:     {len(all_labels_b):,}")

# Per-class accuracy
rows = []
for idx in sorted(set(all_labels_b)):
    cl_id = idx_to_cl[idx]
    name = cl_to_name.get(cl_id, cl_id)
    mask_idx = all_labels_b == idx
    n = mask_idx.sum()
    acc = (all_preds_b[mask_idx] == all_labels_b[mask_idx]).mean()
    rows.append({"cell_type": name, "cl_id": cl_id, "n": n, "accuracy": acc})

df_eval_b = pd.DataFrame(rows).sort_values("accuracy")
print(f"\nPer-class accuracy:")
print(df_eval_b.to_string(index=False, float_format="{:.3f}".format))

# Save checkpoint
checkpoint_path_b = checkpoint_dir / f"mode_b_{run_timestamp}.pt"
state_dict_b = model_b.module.state_dict() if isinstance(model_b, nn.DataParallel) else model_b.state_dict()
checkpoint_b = {
    "epoch": EPOCHS_B,
    "model_state_dict": state_dict_b,
    "optimizer_state_dict": optimizer_b.state_dict(),
    "loss_history": loss_history_b,
    "epoch_times": epoch_times_b,
    "input_dim": embed_dim,
    "output_dim": len(leaf_values),
    "date_preprocessed": DATE,
    "root_cl_id": ROOT_CL_ID,
    "model_class": "ScipherModel",
    "leaf_accuracy": acc_b,
    "macro_f1": macro_f1_b,
    "num_gpus": num_gpus,
}
torch.save(checkpoint_b, checkpoint_path_b)
print(f"\nSaved Mode B checkpoint: {checkpoint_path_b}")

## 4. Evaluation & Comparison

In [ ]:
print("=" * 60)
print("Mode A vs Mode B Comparison")
print("=" * 60)

comparison = pd.DataFrame({
    "Metric": ["Leaf Accuracy", "Macro F1", "Weighted F1", "Epochs", "Total Train Time (s)"],
    "Mode A (WeightedSum+SimpleNN)": [
        f"{acc_a:.4f}", f"{macro_f1_a:.4f}", f"{weighted_f1_a:.4f}",
        EPOCHS_A, f"{sum(epoch_times_a):.1f}",
    ],
    "Mode B (Attention+WideNN)": [
        f"{acc_b:.4f}", f"{macro_f1_b:.4f}", f"{weighted_f1_b:.4f}",
        EPOCHS_B, f"{sum(epoch_times_b):.1f}",
    ],
})
print(comparison.to_string(index=False))

# Per-class comparison
df_compare = df_eval_a[["cell_type", "n", "accuracy"]].rename(
    columns={"accuracy": "acc_a"}
).merge(
    df_eval_b[["cell_type", "accuracy"]].rename(columns={"accuracy": "acc_b"}),
    on="cell_type",
)
df_compare["diff (B-A)"] = df_compare["acc_b"] - df_compare["acc_a"]
df_compare = df_compare.sort_values("diff (B-A)", ascending=False)

print(f"\nPer-class accuracy comparison:")
print(df_compare.to_string(index=False, float_format="{:.3f}".format))

n_b_wins = (df_compare["diff (B-A)"] > 0).sum()
n_a_wins = (df_compare["diff (B-A)"] < 0).sum()
n_ties = (df_compare["diff (B-A)"] == 0).sum()
print(f"\nMode B wins: {n_b_wins}, Mode A wins: {n_a_wins}, Ties: {n_ties}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

def smooth(values, window=50):
    kernel = np.ones(window) / window
    return np.convolve(values, kernel, mode="valid")

for ax, key, title in zip(
    axes, ["total", "leaf", "parent"],
    ["Total Loss", "Leaf Loss", "Parent Loss"],
):
    if len(loss_history_a[key]) > 50:
        ax.plot(smooth(loss_history_a[key]), label="Mode A", alpha=0.8)
    else:
        ax.plot(loss_history_a[key], label="Mode A", alpha=0.8)
    if len(loss_history_b[key]) > 50:
        ax.plot(smooth(loss_history_b[key]), label="Mode B", alpha=0.8)
    else:
        ax.plot(loss_history_b[key], label="Mode B", alpha=0.8)
    ax.set_title(title)
    ax.set_xlabel("Batch")
    ax.set_ylabel("Loss")
    ax.legend()

plt.tight_layout()
plt.savefig(DATA_DIR / "reports" / "training_loss_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to data/reports/training_loss_curves.png")

In [ ]:
import scanpy as sc

sc.settings.set_figure_params(dpi=100, frameon=False)

# Build AnnData for UMAP visualization (val set only)
val_ct = val_adata.obs["cell_type"].values
adata_umap = sc.AnnData(
    X=np.zeros((len(val_ct), 1)),  # placeholder
    obs=pd.DataFrame({"cell_type": val_ct}),
)
adata_umap.obsm["X_ws"] = val_embeddings
adata_umap.obsm["X_attn"] = val_embeddings_b

# UMAP from weighted-sum embeddings (Mode A)
sc.pp.neighbors(adata_umap, use_rep="X_ws")
sc.tl.umap(adata_umap)
adata_umap.obsm["X_umap_ws"] = adata_umap.obsm["X_umap"].copy()

# UMAP from attention embeddings (Mode B)
sc.pp.neighbors(adata_umap, use_rep="X_attn")
sc.tl.umap(adata_umap)
adata_umap.obsm["X_umap_attn"] = adata_umap.obsm["X_umap"].copy()

fig, axes = plt.subplots(1, 2, figsize=(20, 8))

adata_umap.obsm["X_umap"] = adata_umap.obsm["X_umap_ws"]
sc.pl.umap(
    adata_umap, color="cell_type", ax=axes[0], show=False,
    title="Mode A: Weighted-Sum Embeddings",
    legend_loc="on data", legend_fontsize=6, frameon=False,
)

adata_umap.obsm["X_umap"] = adata_umap.obsm["X_umap_attn"]
sc.pl.umap(
    adata_umap, color="cell_type", ax=axes[1], show=False,
    title="Mode B: Attention-Pooled Embeddings",
    legend_loc="on data", legend_fontsize=6, frameon=False,
)

plt.tight_layout()
plt.savefig(
    DATA_DIR / "reports" / "train_embedding_umap_comparison.png",
    dpi=150, bbox_inches="tight",
)
plt.show()
print("Saved to data/reports/train_embedding_umap_comparison.png")